In [166]:
import pandas as pd
import numpy as np
import psycopg

In [167]:
conn = psycopg.connect(
    "dbname=dailyedge_development"
)

In [168]:
query = """
SELECT timestamp, open, high, low, close, volume
FROM CANDLES
WHERE timestamp >= %s
  AND timestamp < %s
  AND timestamp::time >= '08:30:00'
  AND timestamp::time <= '15:15:00'
ORDER BY timestamp
"""

df = pd.read_sql(
    query,
    conn,
    params=("2025-09-01", "2026-07-08")
)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df.shape

/tmp/ipykernel_105386/770822628.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


(87179, 6)

In [169]:
TRAIL_DISTANCE = 50
SCALE_OUT_TARGETS = [50, 100, 150]

In [170]:
sessions = {
    session_date: session.reset_index(drop=True)
    for session_date, session in df.groupby(df["timestamp"].dt.date)
}

len(sessions)

219

In [171]:
def evaluate_scale_out_session(session, direction):
    entry_price = session.iloc[0]["open"]

    if direction == "Long":
        stop = entry_price - TRAIL_DISTANCE
        targets = [entry_price + distance for distance in SCALE_OUT_TARGETS]
        favorable_extreme = entry_price
    elif direction == "Short":
        stop = entry_price + TRAIL_DISTANCE
        targets = [entry_price - distance for distance in SCALE_OUT_TARGETS]
        favorable_extreme = entry_price
    else:
        raise ValueError("direction must be 'Long' or 'Short'")

    active_units = 3
    next_target = 0
    total_pnl = 0.0

    for _, candle in session.iterrows():
        candle_open = candle["open"]
        candle_high = candle["high"]
        candle_low = candle["low"]

        # Stop entering this candle is fixed for the entire candle.
        entering_stop = stop

        # 1. Check candle open against entering stop.
        if direction == "Long" and candle_open <= entering_stop:
            total_pnl += active_units * (entering_stop - entry_price)
            active_units = 0
            break

        if direction == "Short" and candle_open >= entering_stop:
            total_pnl += active_units * (entry_price - entering_stop)
            active_units = 0
            break

        # 2. Determine whether entering stop was touched.
        if direction == "Long":
            stop_touched = candle_low <= entering_stop
        else:
            stop_touched = candle_high >= entering_stop

        # Determine all still-active scale-out targets touched this candle.
        touched_targets = []

        for i in range(next_target, len(targets)):
            if direction == "Long" and candle_high >= targets[i]:
                touched_targets.append(i)
            elif direction == "Short" and candle_low <= targets[i]:
                touched_targets.append(i)
            else:
                break

        # Stop and any new target touched in same candle:
        # 1-minute OHLC cannot establish ordering.
        if stop_touched and touched_targets:
            return {
                "status": "Unknown",
                "total_pnl": np.nan,
            }

        # Stop only.
        if stop_touched:
            if direction == "Long":
                total_pnl += active_units * (entering_stop - entry_price)
            else:
                total_pnl += active_units * (entry_price - entering_stop)

            active_units = 0
            break

        # Scale out each target reached.
        for i in touched_targets:
            target_price = targets[i]

            if direction == "Long":
                total_pnl += target_price - entry_price
            else:
                total_pnl += entry_price - target_price

            active_units -= 1
            next_target = i + 1

        if active_units == 0:
            break

        # 3. Trade survived candle: update trailing stop ONCE.
        if direction == "Long":
            favorable_extreme = max(favorable_extreme, candle_high)
            stop = max(stop, favorable_extreme - TRAIL_DISTANCE)
        else:
            favorable_extreme = min(favorable_extreme, candle_low)
            stop = min(stop, favorable_extreme + TRAIL_DISTANCE)

    # Flatten remaining units at the final RTH close.
    if active_units > 0:
        final_close = session.iloc[-1]["close"]

        if direction == "Long":
            total_pnl += active_units * (final_close - entry_price)
        else:
            total_pnl += active_units * (entry_price - final_close)

    return {
        "status": "Resolved",
        "total_pnl": total_pnl,
    }

In [172]:
def get_first_to_target_direction(session, target_distance=105):
    entry_price = session.iloc[0]["open"]

    long_target = entry_price + target_distance
    short_target = entry_price - target_distance

    for _, candle in session.iterrows():
        long_hit = candle["high"] >= long_target
        short_hit = candle["low"] <= short_target

        if long_hit and short_hit:
            return "Ambiguous"

        if long_hit:
            return "Long"

        if short_hit:
            return "Short"

    return "Neither"

In [173]:
correct_results = []

for session_date, session in sessions.items():
    direction = get_first_to_target_direction(session, 105)

    if direction in ("Long", "Short"):
        result = evaluate_scale_out_session(session, direction)

        correct_results.append({
            "date": session_date,
            "direction": direction,
            "status": result["status"],
            "total_pnl": result["total_pnl"],
        })

correct_results_df = pd.DataFrame(correct_results)

correct_results_df["status"].value_counts()

status
Resolved    204
Unknown       1
Name: count, dtype: int64

In [174]:
resolved_correct = correct_results_df[
    correct_results_df["status"] == "Resolved"
]

print("Resolved trades:", len(resolved_correct))
print("Average P&L:", resolved_correct["total_pnl"].mean())
print("Median P&L:", resolved_correct["total_pnl"].median())
print("Winning trades:", (resolved_correct["total_pnl"] > 0).sum())
print("Losing trades:", (resolved_correct["total_pnl"] < 0).sum())
print("Breakeven trades:", (resolved_correct["total_pnl"] == 0).sum())

Resolved trades: 204
Average P&L: 68.67334414215705
Median P&L: 75.11243800000375
Winning trades: 120
Losing trades: 84
Breakeven trades: 0


In [175]:
wrong_results = []

for _, row in correct_results_df.iterrows():
    if row["status"] != "Resolved":
        continue

    session = sessions[row["date"]]

    wrong_direction = (
        "Short" if row["direction"] == "Long"
        else "Long"
    )

    result = evaluate_scale_out_session(session, wrong_direction)

    wrong_results.append({
        "date": row["date"],
        "direction": wrong_direction,
        "status": result["status"],
        "total_pnl": result["total_pnl"],
    })

wrong_results_df = pd.DataFrame(wrong_results)

wrong_results_df["status"].value_counts()

status
Resolved    200
Unknown       4
Name: count, dtype: int64

In [176]:
matched = (
    resolved_correct[["date", "total_pnl"]]
    .merge(
        wrong_results_df[
            wrong_results_df["status"] == "Resolved"
        ][["date", "total_pnl"]],
        on="date",
        suffixes=("_correct", "_wrong")
    )
)

print("Matched sessions:", len(matched))
print("Average correct P&L:", matched["total_pnl_correct"].mean())
print("Average wrong P&L:", matched["total_pnl_wrong"].mean())
print("Median correct P&L:", matched["total_pnl_correct"].median())
print("Median wrong P&L:", matched["total_pnl_wrong"].median())

Matched sessions: 200
Average correct P&L: 72.35483573500018
Average wrong P&L: -63.89042283000006
Median correct P&L: 78.8457210000015
Median wrong P&L: -89.52325800000108


In [177]:
avg_correct = matched["total_pnl_correct"].mean()
avg_wrong = matched["total_pnl_wrong"].mean()

accuracies = np.arange(0.50, 0.81, 0.05)

expectancy = pd.DataFrame({
    "Accuracy": accuracies,
    "Expectancy": [
        accuracy * avg_correct + (1 - accuracy) * avg_wrong
        for accuracy in accuracies
    ]
})

expectancy

,Accuracy,Expectancy
0,0.50,4.232206
1,0.55,11.044469
2,0.60,17.856732
3,0.65,24.668995
4,0.70,31.481258
5,0.75,38.293521
6,0.80,45.105784


In [178]:
for label, column in [
    ("Correct", "total_pnl_correct"),
    ("Wrong", "total_pnl_wrong"),
]:
    pnl = matched[column]

    print(label)
    print("  Winners:", (pnl > 0).sum())
    print("  Losers:", (pnl < 0).sum())
    print("  Average winner:", pnl[pnl > 0].mean())
    print("  Average loser:", pnl[pnl < 0].mean())
    print()

Correct
  Winners: 120
  Losers: 80
  Average winner: 177.11495526666658
  Average loser: -84.78534356249943

Wrong
  Winners: 39
  Losers: 161
  Average winner: 95.97600169230692
  Average loser: -102.61583001242225



In [179]:
correct = matched["total_pnl_correct"]
wrong = matched["total_pnl_wrong"]

correct_win_rate = (correct > 0).mean()
correct_loss_rate = (correct < 0).mean()

wrong_win_rate = (wrong > 0).mean()
wrong_loss_rate = (wrong < 0).mean()

correct_avg_win = correct[correct > 0].mean()
correct_avg_loss = correct[correct < 0].mean()

wrong_avg_win = wrong[wrong > 0].mean()
wrong_avg_loss = wrong[wrong < 0].mean()

rows = []

for accuracy in accuracies:
    expected_win_rate = (
        accuracy * correct_win_rate
        + (1 - accuracy) * wrong_win_rate
    )

    expected_loss_rate = (
        accuracy * correct_loss_rate
        + (1 - accuracy) * wrong_loss_rate
    )

    avg_winner = (
        accuracy * correct_win_rate * correct_avg_win
        + (1 - accuracy) * wrong_win_rate * wrong_avg_win
    ) / expected_win_rate

    avg_loser = (
        accuracy * correct_loss_rate * correct_avg_loss
        + (1 - accuracy) * wrong_loss_rate * wrong_avg_loss
    ) / expected_loss_rate

    rows.append({
        "Accuracy": accuracy,
        "Win Rate": expected_win_rate,
        "Avg Winner": avg_winner,
        "Avg Loser": avg_loser,
        "Reward/Risk": avg_winner / abs(avg_loser),
    })

rr_by_accuracy = pd.DataFrame(rows)

rr_by_accuracy

,Accuracy,Win Rate,Avg Winner,Avg Loser,Reward/Risk
0,0.50,0.39750,157.212948,-96.696996,1.625831
1,0.55,0.41775,160.071405,-95.878678,1.669520
2,0.60,0.43800,162.665553,-95.001387,1.712244
3,0.65,0.45825,165.030430,-94.058513,1.754551
4,0.70,0.47850,167.195146,-93.042414,1.796978
5,0.75,0.49875,169.184080,-91.944217,1.840073
6,0.80,0.51900,171.017808,-90.753552,1.884420
